In [2]:
from Crypto.Util.number import *
from sage.all import *

flag = b'nex{this_is_test}'
m = bytes_to_long(flag)
p = getPrime(512)
q = getPrime(512)
e = getPrime(1250)
phi = (p-1)*(q-1)
d = inverse_mod(e,phi)
n = p*q

c = pow(m,e,n)
k = (e*d-1)//phi
print(int(k%e).bit_length())
print(int((p+q)%e).bit_length())

1248
513


In [3]:
# e*d == 1 + k*(p*q - (p+q) + 1) = 1+k*(n - s + 1) = 0 mod  N
PR = PolynomialRing(Zmod(e), names=('x', 'y'))
x, y = PR.gens()
f = 1 + x*(n - y + 1)
f(x=k,y=p+q)

0

In [4]:
import itertools
from multiprocessing import Pool, cpu_count
def small_roots(f, bounds, m=1, d=None):
    if not d:
        d = f.degree()

    R = f.base_ring()
    N = R.cardinality()

    f /= f.coefficients().pop(0)
    f = f.change_ring(ZZ)

    G = Sequence([], f.parent())
    for i in range(m + 1):
        base = N ** (m - i) * f ** i
        for shifts in itertools.product(range(d), repeat=f.nvariables()):
            g = base * prod(map(power, f.variables(), shifts))
            G.append(g)

    B, monomials = G.coefficients_monomials()
    monomials = vector(monomials)

    factors = [monomial(*bounds) for monomial in monomials]
    for i, factor in enumerate(factors):
        B.rescale_col(i, factor)

    B = B.dense_matrix().LLL()

    B = B.change_ring(QQ)
    for i, factor in enumerate(factors):
        B.rescale_col(i, 1 / factor)

    H = Sequence([], f.parent().change_ring(QQ))
    for h in filter(None, B * monomials):
        H.append(h)
        I = H.ideal()
        if I.dimension() == -1:
            H.pop()
        elif I.dimension() == 0:
            roots = []
            for root in I.variety(ring=ZZ):
                root = tuple(R(root[var]) for var in f.variables())
                roots.append(root)
            return roots

    return []
small_roots(f,[2**1249,2**513],m=4,d=3)

[]

In [5]:
from Crypto.Util.number import *
import os

proof.arithmetic(False)  # to make sage faster

flag = b"TSJ{not_real_flag}"

p = getPrime(1024)
q = getPrime(512)
n = p * q
e = 65537
E = EllipticCurve(Zmod(n), [p, q])

while True:
    x = ZZ(bytes_to_long(flag + os.urandom(192 - len(flag))))
    try:
        yp = ZZ(E.change_ring(GF(p)).lift_x(x).xy()[1])
        yq = ZZ(E.change_ring(GF(q)).lift_x(x).xy()[1])
        y = crt([yp, yq], [p, q])
        break
    except:
        pass

C = e * E(x, y)
print(n)
print(C.xy())

1480152987740988472806226340981840614189471709765819580379876812563580104390372925326798160736502571651204528073094525829938514333398776932542382603058161263334772486341593299959326730062375802475382875875421318479311370417791708452484543494643485225307263585628998929238821374370665448640241803627970407802510890942103350528725100290044878455943487015448507190100266595525757924310727130459268508609210603320511037073110200108036596585124727002566196207382422599
(470430816994267528473241733986426813312733512117502968217178074878828936506786061373742973588636828608919000891258427494913022260338935126824612955502142978246450427933810210594042778240010799240509571720056333792364296372093220801972793219689521207440580034699037917002316383736452257938418945827252428364832999340576274582641571909455000867198423856527583175981994428363797551640350642494508054662206242263811304947989971956103109523864252514727385367004887116, 13296585216436873316951653628008970017893119994172149773934794517869726

In [9]:
from sage.all import *
from Crypto.Util.number import *
e = 65537
n = 1084688440161525456565761297723021343753253859795834242323030221791996428064155741632924019882056914573754134213933081812831553364457966850480783858044755351020146309359045120079375683828540222710035876926280456195986410270835982861232693029200103036191096111928833090012465092747472907628385292492824489792241681880212163064150211815610372913101079146216940331740232522884290993565482822803814551730856710106385508489039042473394392081462669609250933566332939789
cx,cy = (1079311510414830031139310538989364057627185699077021276018232243092942690870213059161389825534830969580365943449482350229248945906866520819967957236255440270989833744079711900768144840591483525815244585394421988274792758875782239418100536145352175259508289748680619234207733291893262219468921233103016818320457126934347062355978211746913204921678806713434052571635091703300179193823668800062505275903102987517403501907477305095029634601150501028521316347448735695, 950119069222078086234887613499964523979451201727533569872219684563725731563439980545934017421736344519710579407356386725248959120187745206708940002584577645674737496282710258024067317510208074379116954056479277393224317887065763453906737739693144134777069382325155341867799398498938089764441925428778931400322389280512595265528512337796182736811112959040864126090875929813217718688941914085732678521954674134000433727451972397192521253852342394169735042490836886)
PR = PolynomialRing(Zmod(n), names=('q'))
q = PR.gen()
f = cx**3 + q - cy**2
q = ZZ(f.monic().small_roots(X=2**512,beta=0.6)[0])
p = n//q
E = EllipticCurve(Zmod(n), [p, q])
C = E(cx,cy)
phi = E.change_ring(GF(p)).order() * E.change_ring(GF(q)).order()
M = C*inverse_mod(e,phi)
long_to_bytes(int(M[0]))

b"TSJ{i_don't_know_how_to_come_up_with_a_good_flag_sorry}S+V\xd8-\x9cQ9\x07\xb0\xdb\xd4h\x1d\x08\xa9=\xc4\x97\xa1\xd1\xa8\x9e\x82\xf5\x85iq\xc4\x03\x9a\xa55\xcb\xc4\x18I\xc5so\xafhHPk\x1cZ\xdf\x00P\xfc9X\xdep\xb1\xe6\x85\xdd\xb6\x07%\xbb!\xbc\xf2M\xf8\x1b\xdd\xf3\xed\x06\xf8\xe3-\xa53NWb\xc5\xa2\xd6;o\xc9\x0fMtI\xf3\xa2\x8bU\xa4>\x15~\xb1\xc2\xbc\x85\x8dM\x06\xdc>\xd1\xa2\xa2\xb9cwN\xa5)\x0c=\xacF\xe0[\xb9\xbdgcIYk\xbd\xfe\xde\xd1\xe0\xb7\xfe"